# Milestone 4 exact-token TP crossover shard

Status: **PREPARED_FOR_KAGGLE**. Use a brand-new T4 x2 session with Internet enabled. Configure private Kaggle secrets `HF_TOKEN` and `M4_SHARD_ID`; choose a shard ID verbatim from `research/M4_EXECUTION_PLAN.json`. One notebook execution runs one model only and preserves unsupported/OOM outcomes.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

EXPECTED_SOURCE_COMMIT = 'SOURCE_COMMIT_TO_PIN_AFTER_REVIEW'
WORK = Path('/kaggle/working'); SOURCE = WORK/'kaggle-vllm-source'; RUNTIME = WORK/'kaggle-vllm-runtime'; CACHE = WORK/'kaggle-vllm-cache'; OUTPUT_ROOT = WORK/'m4-evidence'; HF_HOME = WORK/'hf-cache'
assert Path('/kaggle').is_dir() and not SOURCE.exists() and not RUNTIME.exists() and not OUTPUT_ROOT.exists(), 'Use a fresh Kaggle session'
secrets = UserSecretsClient(); token = secrets.get_secret('HF_TOKEN'); shard_id = secrets.get_secret('M4_SHARD_ID')
assert token and shard_id, 'Configure HF_TOKEN and M4_SHARD_ID Kaggle secrets'; os.environ['HF_TOKEN'] = token; os.environ['HF_HOME'] = str(HF_HOME)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'kaggle-vllm[hub]==0.2.0'], check=True)
subprocess.run(['git', 'init', str(SOURCE)], check=True); subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', 'https://github.com/kaggle-vllm/kaggle-vllm.git'], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', EXPECTED_SOURCE_COMMIT], check=True); subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
head = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip(); dirty = subprocess.check_output(['git', '-C', str(SOURCE), 'status', '--porcelain'], text=True).strip(); assert head == EXPECTED_SOURCE_COMMIT and not dirty
plan = json.loads((SOURCE/'research/M4_EXECUTION_PLAN.json').read_text()); entries = plan['compatibility_order'] + plan['principal_order']; matches = [item for item in entries if item['shard_id'] == shard_id]; assert len(matches) == 1, 'M4_SHARD_ID is not in the reviewed execution plan'; shard = matches[0]
RUNTIME.mkdir(); CACHE.mkdir(); HF_HOME.mkdir(); manifest = RUNTIME/'runtime.json'
boot = ['kaggle-vllm', 'bootstrap', '--strict', '--staged', str(RUNTIME/'staged'), '--overlay', str(RUNTIME/'overlay'), '--cache', str(CACHE), '--manifest', str(manifest)]
subprocess.run(boot + ['--dry-run'], check=True); subprocess.run(boot, check=True)
runtime = json.loads(manifest.read_text()); RUN_ENV = dict(os.environ); RUN_ENV.update(runtime['runtime_environment']); RUN_ENV['PYTHONPATH'] = str(SOURCE/'src') + os.pathsep + str(SOURCE) + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
print(json.dumps({'source_commit': head, 'shard': shard, 'token_present_not_printed': True}, indent=2))

In [ ]:
output = OUTPUT_ROOT/f"{shard['model_key']}-{shard['workload']}-r{shard['repetition']:02d}-{shard['mode']}"
command = [sys.executable, str(SOURCE/'scripts/kaggle_m4_multimodel_crossover.py'), '--repository', str(SOURCE), '--output-root', str(OUTPUT_ROOT), '--model-key', shard['model_key'], '--workload', shard['workload'], '--repetition', str(shard['repetition']), '--mode', shard['mode'], '--source-identity', EXPECTED_SOURCE_COMMIT]
completed = subprocess.run(command, cwd=SOURCE, env=RUN_ENV, check=False); assert output.is_dir(), 'No evidence directory was preserved'
subprocess.run([sys.executable, '-m', 'kaggle_vllm.research', 'verify-hashes', str(output)], cwd=SOURCE, env=RUN_ENV, check=True)
archive = Path(shutil.make_archive(str(output), 'zip', output)); digest = hashlib.sha256(archive.read_bytes()).hexdigest(); print(json.dumps({'runner_returncode': completed.returncode, 'evidence': str(output), 'zip': str(archive), 'zip_sha256': digest}, indent=2))
assert completed.returncode in {0, 2, 3, 4}, 'Unexpected runner failure; preserve notebook output for review'

Download the printed ZIP and this executed notebook. A zero return code is only a **canonical candidate**, not accepted evidence. Return codes 2 or 3 preserve compatibility/resource-gate evidence and must not be relabeled as a scientific result.